# Imports

In [ ]:
import requests
import pandas as pd
import time
import os
from difflib import SequenceMatcher
import unicodedata


# Formalia

Please read the [assignment overview page](https://laura.alessandretti.com/comsocsci2026/wiki_pages/Assignments.html) carefully before proceeding. The page contains information about formatting (including formats etc), group sizes, and many other aspects of handing in the assignment. 

__If you fail to follow these simple instructions, it will negatively impact your grade!__

**Due date and time**: The assignment is due on Mar 3rd at 23:59. Hand in your Jupyter notebook file (with extension `.ipynb`) via DTU Learn _(Assignment 1)_. 

Remember to include in the first cell of your notebook:
* the link to your group's Git repository 
* group members' contributions


## Part 1: Ready Made vs Custom Made Data

> **Exercise: Ready made data vs Custom made data** In this exercise, I want to make sure you have understood they key points of my lecture and the reading. 
>
> 1. What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book __(answer in max 150 words)__.


Centola’s randomized experiment supports a clean causal claim: differences in adoption can be attributed to network structure, not pre-existing similarity. Trade-offs are response bias (people may act differently when they know they’re studied), a narrow/low-stakes behavior, a small sample, and an artificial setup that may not generalize.

Nicolaides’s ready-made observational data avoids response bias and captures real-world behavior at massive scale. But it can’t nail causality: patterns could be social influence, homophily, or shared context (weather, local events), and platform features can introduce algorithmic confounding. Controls help, scale boosts power and coverage, but it doesn’t fix identification—and there are also consent/privacy ethics concerns.

> 2. How do you think these differences can influence the interpretation of the results in each study? __(answer in max 150 words)__

Centola’s randomized experiment supports a clean causal claim: differences in adoption can be attributed to network structure, not pre-existing similarity. The trade-off is generalizability—it's a controlled, artificial setting and a narrow, low-stakes behavior, so people may doubt it carries over to real networks.

Nicolaides’s real-world observational data is more natural and broadly applicable, but harder to interpret. Similar running patterns could be social influence, homophily, or shared context (weather, local events). Controls and scale help, yet the identification problem remains, and platform effects can amplify bias.

## Part 2: Find Researchers using the OpenAlex API

> **Exercise 3: Find potential Computational Social Scientists** In this exercise, we'll use the OpenAlex API to compile a list of researchers in the field of Computational Social Science, focusing on those who have attended the IC2S2 conference in 2025. This will not only later on help you understand the landscape of Computational Social Science research but also develop practical skills in data collection and analysis. 
>
> Please read the text of the whole exercise before starting to work on it. 
>
> **Steps**
> 
> 1. **Retreive data.** Consider the set of unique researcher names that you collected in Week 1, Exercise 3. Use the _authors_ endpoint of the [OpenAlex API](https://docs.openalex.org/api-entities/authors) to _search_ these researchers in the database based on their names. Loop through the list and, for each researcher in your list, find: 
>     - their _id_: The OpenAlex ID for this author.
>     - their _display\_name_: The name of the author as a single string.
>     - their _works\_api\_url_: A URL that will get you a list of all this author's works.
>     - their _h\_index_ : The h-index for this author.
>     - their _works\_count_: The number of  Works this this author has created.
>     - their _country\_code_: The country code of their last known institution
> 2. **Data Storage** Store this information in a Pandas DataFrame and save it to file.
>


In [ ]:
# Load researcher names collected in Week 1
names = pd.read_csv("researchers.csv")["name"].tolist()

# Insert Api key from .env file
API_KEY = os.getenv("OPENALEX_API_KEY")

results = []

for i, name in enumerate(names):
    r = requests.get(
        "https://api.openalex.org/authors",
        params={"search": name, "per-page": 1, "api_key": API_KEY}
    )
    data = r.json().get("results", [])

    if data:
        a = data[0]
        # last_known_institutions is a list; take first entry
        institutions = a.get("last_known_institutions") or [{}]
        results.append({
            "search_name":   name,
            "id":            a.get("id"),
            "display_name":  a.get("display_name"),
            "works_api_url": a.get("works_api_url"),
            "h_index":       a.get("summary_stats", {}).get("h_index"),
            "works_count":   a.get("works_count"),
            "country_code":  institutions[0].get("country_code"),
        })
    else:
        results.append({"search_name": name})

    if i % 100 == 0:
        print(f"{i}/{len(names)} done...")

    time.sleep(0.05)  # stay under rate limit

# Save to file D1
df_d1 = pd.DataFrame(results)
df_d1.to_csv("D1.csv", index=False)
print(f"{df_d1['id'].notna().sum()}/{len(df_d1)} researchers found in OpenAlex")
df_d1.head()

In [ ]:
df = pd.read_csv("D1.csv")

def last_name(name):
    if pd.isna(name) or str(name).strip() == "": return None
    normalized = unicodedata.normalize("NFD", name).encode("ascii", "ignore").decode("ascii").strip()
    if not normalized: return None
    return normalized.split()[-1].lower()

df = df[df["display_name"].notna()]
df = df[df.apply(lambda r: last_name(r["search_name"]) == last_name(r["display_name"]), axis=1)]
df = df.reset_index(drop=True)

print(f"Rows after cleaning: {len(df)}")
df.to_csv("D1_clean.csv", index=False)

>    
> **Handling Challenges**
> 
> I expect that, while working on the steps above, you will encounter several obstacles. As you complete this exercise, you are expected to:     
>
>    - Identify problems that arise.      
>    - Improve your solutions to address such problems, making reasonable decisions when data is incomplete or ambiguous.       
>
> **Reflection Questions**
>  Answer the following questions __(max 150 words for each question)__: 
>
>    - Which challenges did you encounter? How did you address them?

>    - Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data? 

## Part 3: Collect Research Articles

> **Exercise 1: Collecting Research Articles from IC2S2 Authors**
>
>In this exercise, we'll leverage the OpenAlex API to gather information on research articles authored by participants of the IC2S2 2025 conference, referred to as *IC2S2 authors*. **Before you start, please ensure you read through the entire exercise.**
>
> 
> **Steps:**
>  
> 1. **Retrieve Data:** Start with the dataset of *IC2S2 authors* you collected in Week 2, Exercise 3 (called dataset D1 in the figure above). Use the OpenAlex API [works endpoint](https://docs.openalex.org/api-entities/works) to fetch their research articles. For each article, retrieve the following details:
>    - _id_: The unique OpenAlex ID for the work.
>    - _publication_year_: The year the work was published.
>    - _cited_by_count_: The number of times the work has been cited by other works.
>    - _author_ids_: The OpenAlex IDs for the authors of the work.
>    - _title_: The title of the work.
>    - _abstract_inverted_index_: The abstract of the work, formatted as an inverted index.
> 
>     **Important Note on Paging:** By default, the OpenAlex API limits responses to 25 works per request. For more efficient data retrieval, I suggest to adjust this limit to 200 works per request. Even with this adjustment, you will need to implement pagination to access all available works for a given query. This ensures you can systematically retrieve the complete set of works beyond the initial 200. Find guidance on implementing pagination [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/paging#cursor-paging).
>
> 2. **Data Storage:** Organize the retrieved information into two Pandas DataFrames and save them to two files in a suitable format:
>    - Dataset D2: The *IC2S2 papers* dataset should include: *id, publication\_year, cited\_by\_count, author\_ids*.
>    - Dataset D3: The *IC2S2 abstracts* dataset should include: *id, title, abstract\_inverted\_index*.
>  
>
> **Filters:**
> To ensure the data we collect is relevant and manageable, apply the following filters:
>     
>    - Only include *IC2S2 authors* with a total work count between 5 and 5,000.    
>    - Retrieve only works that have received more than 10 citations.    
>    - Limit to works authored by fewer than 10 individuals.    
>    - Include only works relevant to Computational Social Science (focusing on: Sociology OR Psychology OR Economics OR Political Science) AND intersecting with a quantitative discipline (Mathematics OR Physics OR Computer Science), as defined by their [Concepts](https://docs.openalex.org/api-entities/works/work-object#concepts). *Note*: here we only consider Concepts at *level=0* (the most coarse definition of concepts).     
>
> **Efficiency Tips:**
> Writing efficient code in this exercise is **crucial**. To speed up your process:
> 
> - **Apply filters directly in your request:** When possible, use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) of the *works* endpoint to apply the filters above directly in your API request, ensuring only relevant data is returned. Learn about combining multiple filters [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).  
> - **Bulk requests:** Instead of sending one request for each author, you can use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) to query works by multiple authors in a single request. *Note: My testing suggests that can only include up to 25 authors per request.*
> - **Use multiprocessing:** Implement multiprocessing to handle multiple requests simultaneously. I highly recommmend [Joblib’s Parallel](https://joblib.readthedocs.io/en/stable/) function for that, and [tqdm](https://tqdm.github.io/) can help monitor progress of your jobs. Remember to stay within [the rate limit](https://docs.openalex.org/how-to-use-the-api/rate-limits-and-authentication) of 100 requests per second.
>
>
>   
> For reference, employing these strategies allowed me to fetch the data in about 30 seconds using 5 cores on my laptop. I obtained a dataset of approximately 25 MB (including both the *IC2S2 abstracts* and *IC2S2 papers* files).
> 
>
> **Data Overview and Reflection questions:** Answer the following questions __(max 150 words for each question)__:     

> - **Dataset summary.** How many works are listed in your Dataset D2 (*IC2S2 papers*) dataframe? How many unique researchers have co-authored these works?     

> - **Efficiency in code.** Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?  

> - **Filtering Criteria and Dataset Relevance** Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?  